# Fujisawa, Amarasingham, Harrison and Buzsaki (2008) on CRCNS pfc-2

Paper: Fujisawa S, Amarasingham A, Harrison MT, Buzsaki G. Behavior-dependent short-term
assembly dynamics in the medial prefrontal cortex. Nature Neuroscience 11:823-833 (2008),
doi:10.1038/nn.2134. The claims file, `claims/crcns-pfc-2/10.1038-nn.2134.yaml`, was written from
the version of record with its supplement.

Dataset: CRCNS pfc-2, doi:10.6080/K01V5BWK (dataset `crcns:pfc-2/unversioned`). CRCNS data sets
are not versioned (docs/archives.md), so the version pinned is `unversioned`: the files as they
are in the bucket `crcnsarchive` under `pfc-2/live/`, the only published copy, documented by the
data description version 0.7 (2021). The MD5 sums of the files read, from the data set's
`checksums.md5`, are printed below and compared with the bytes read, so that a later change is
detectable. Files read: `data/behaviors.zip`, `data/EE0708/EE188_Behavior.mat`,
`data/EE0708/EE188_Behavior_PFC.mat`, and the member `EE188_example.mat` of
`data/EE0708/EE.188.tar.gz`.

Claims (outcomes are in the attempt record and the summary at the end).

| Claim | Tested here |
|---|---|
| unit-count | tested |
| task-performance | blocked: the data hold correct trials only |
| fig2c-synchrony-by-layer | tested |
| fig2-population-uniform | tested (layer 2/3 session only; the layer 5 session is not in the data set) |
| lifetime-width | tested |
| trajectory-divergence | tested |
| side-arm-selectivity-by-layer | tested |
| central-arm-selectivity | tested |
| odor-selective-fraction | tested |
| odor-unrelated-to-stem | tested |
| retrospective-coding-rare | tested |
| connection-counts | tested |
| cell-type-fractions | tested |
| pyr-pyr-connections | tested |
| fig3b-session-connections | tested (layer 2/3 session only) |
| fig3d-hub | tested (layer 2/3 session only) |
| fig3f-distance | tested |
| nearby-pyr-int-probability | tested |
| connected-cells-trajectory-specific | tested |
| no-spatial-clustering | tested |
| position-dependent-transmission | tested |
| fig5a-session-selective-connections | tested |
| fig4-pair-156-136 | tested |
| fig5b-example-pairs | tested |
| thinning-preserves-dependence | tested |
| fig6-depression-facilitation | tested |
| fig6b-convergent-examples | tested |
| fig7-supralinear-coincidence | tested |
| speed-not-correlated | tested |
| population-vector-width | tested |

In [ ]:
import hashlib
import os
import time

os.environ.setdefault("OMP_NUM_THREADS", "1")
import fsspec
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from scipy import stats

import pfc2

T0 = time.time()
raw = pfc2.read_raw()
with fsspec.open(pfc2.BASE.replace("data/", "checksums.md5")) as f:
    md5 = dict(reversed(line.split()) for line in f.read().decode().splitlines() if line.strip())
with fsspec.open(pfc2.BASE + "behaviors.zip", block_size=2**22) as f:
    h = hashlib.md5()
    while chunk := f.read(2**24):
        h.update(chunk)
print("data/behaviors.zip", md5["data/behaviors.zip"], "read", h.hexdigest())
for k in ["EE188_Behavior.mat", "EE188_Behavior_PFC.mat"]:
    print("data/EE0708/" + k, md5["data/EE0708/" + k], "read", hashlib.md5(raw[k]).hexdigest())
print("data/EE0708/EE.188.tar.gz", md5["data/EE0708/EE.188.tar.gz"], "(member EE188_example.mat read in the stream)")
S = pfc2.load_sessions(raw)
EX = pfc2._mat(raw["EE188_example.mat"])
print("loaded in", round(time.time() - T0), "s")

## Data check

The data set holds 13 working memory sessions of three rats (EE, FF, GG). The paper recorded four
rats; Supplementary Figure 1 names them E, G and H (layer 2/3) and F (layer 5). The data set's
rats EE, FF and GG are taken to be the paper's E, F and G (an assumption: the names match and the
data description names EE.188 as the session of Figures 2a and 3d, which is layer 2/3); rat H is
not in the data set. So FF is the layer 5 rat.

In [ ]:
L23 = [k for k, s in S.items() if s.layer == "L2/3"]
L5 = [k for k, s in S.items() if s.layer == "L5"]
print(f"{'session':8s} {'rat':3s} {'layer':5s} {'cells':>5s} {'pyr/int/unk (file)':>18s} {'trials':>6s} {'L':>3s} {'R':>3s} "
      f"{'spikes':>8s} {'dur s':>6s} {'shanks':>6s}")
for k, s in S.items():
    ty = np.bincount(s.celltype_file, minlength=4)[1:]
    print(f"{k:8s} {s.rat:3s} {s.layer:5s} {s.n:5d} {str(ty):>18s} {s.ntrials:6d} {s.left.sum():3d} {(~s.left).sum():3d} "
          f"{len(s.t):8d} {s.duration:6.0f} {len(np.unique(s.shank)):6d}")
print("cells per rat:", {r: sum(s.n for s in S.values() if s.rat == r) for r in ("EE", "FF", "GG")})
print("sessions per layer:", {"L2/3": L23, "L5": L5})
allx = np.concatenate([s.x[s.x >= 0] for s in S.values()])
print(f"spike positions (spikeX): {allx.min():.4f} to {allx.max():.4f} (normalised length); "
      f"spikes outside trials marked -1: {np.mean(np.concatenate([s.x < 0 for s in S.values()])):.2f} of all")
# trial labels of spikes against the trial table
for k, s in S.items():
    assert s.max_ltr == s.left.sum() and s.max_rtr == (~s.left).sum()
    tt = s.trial >= 0
    st, en = s.NP[s.trial[tt], 0], s.NP[s.trial[tt], 2]
    inside = np.mean((s.t[tt] >= st - 0.05) & (s.t[tt] <= en + 0.05))
    d = s.whl[np.minimum((s.t[tt] * pfc2.WHL_RATE).astype(int), len(s.whl) - 1), 6]
    ok = d > 0
    r = np.corrcoef(s.x[tt][ok], d[ok] - 1)[0, 1]
    print(f"{k}: labelled spikes inside their trial's [nose-poke start, end] (+-50 ms): {inside:.4f}; "
          f"spikeX against the position file at the spike time r = {r:.3f}; trial durations "
          f"{np.min(s.NP[:, 2] - s.NP[:, 1]):.1f} to {np.max(s.NP[:, 2] - s.NP[:, 1]):.1f} s; "
          f"nose-poke {np.min(s.NP[:, 1] - s.NP[:, 0]):.2f} to {np.max(s.NP[:, 1] - s.NP[:, 0]):.1f} s")
w = S["EE188"].whl
print("position file EE188: x1", w[w[:, 0] > 0, 0].min(), "to", w[:, 0].max(), "y1", w[w[:, 1] > 0, 1].min(), "to",
      w[:, 1].max(), "(pixels); d", w[w[:, 6] > 0, 6].min(), "to", w[:, 6].max())

The trial table, `SessionNP`, lists correct trials only (data description, page 2), so the
error trials and the performance are not in the data set. Above: every spike with a trial label
falls in that trial's interval from the first nose-poke to the trial end (so the spikes' trials,
and their positions near 0, include the nose-poke, which the description's trial start, the end of
the nose-poke, does not); the spike positions agree with the position file; the spike times in
20 kHz samples, converted to seconds, end with the position file. The `CellsInfo` type column is described as pyramidal (1), interneuron (2) or
unknown (3): in the 12 sessions of `behaviors.zip` and in `EE188_Behavior_PFC.mat` no cell is
unknown, while `EE188_example.mat` has 33, 15 and 69.

In [ ]:
s = S["EE188"]
p105 = pfc2._mat(raw["EE188_Behavior_PFC.mat"])
c105 = p105["CellsInfo"][:, 0].astype(int)
print("EE188_example.mat cells:", s.n, "types", np.bincount(s.celltype_file, minlength=4)[1:],
      "| EE188_Behavior_PFC.mat cells:", len(c105), "types", np.bincount(p105["CellsInfo"][:, 2].astype(int), minlength=4)[1:])
print("cells of the example not in Behavior_PFC:", sorted(set(s.cells) - set(c105)),
      "| of Behavior_PFC not in the example:", sorted(set(c105) - set(s.cells)))
same = np.isin(s.cells, c105)
t1 = np.sort(np.asarray(p105["spiket"]).ravel())
print("spikes: example", len(s.t), "Behavior_PFC", len(t1), "; example spikes of the shared cells",
      int(np.isin(s.u, np.flatnonzero(same)).sum()))
# pairs of sessions with identical spike counts per cell (a sign of a recording deposited twice)
sig = {k: tuple(sorted(np.bincount(v.u))) for k, v in S.items()}
dups = [(a, b) for a in S for b in S if a < b and sig[a] == sig[b]]
print("sessions with identical spike count lists:", dups if dups else "none")
print("spikeph (all clusters of the day's sorting on the 8 PFC shanks) against the cells kept:")
for k in S:
    if k == "EE188":
        n_ph = p105["spikeph"].shape[0]
    else:
        n_ph = pfc2._mat(raw[k + "_Behavior_PFC.mat"])["spikeph"].shape[0]
    print(f"  {k}: {n_ph} clusters, {S[k].n} cells")

Clock check. Every analysis here uses the spike times and the trial table, and the position file
for the claims on paths, speed and synchrony. To show they share one clock, the trial side (left
or right) is decoded from the population spike counts in the 2 s after each trial start
in the 3 s before each trial end, in the side arm (nearest centroid, leave one out), once with the
true trial times and once with the times shifted by 30 s. The position file is checked by the agreement of `spikeX` with the
file's position at the spike time (above), and by decoding the side from the position file's
x coordinate 2 to 4 s after the trial start (end of the nose-poke) against the same shift.

In [ ]:
def loo_centroid(X, y):
    X = (X - X.mean(0)) / (X.std(0) + 1e-9)
    hit = 0
    for i in range(len(y)):
        m = np.ones(len(y), bool)
        m[i] = False
        c1, c0 = X[m & y].mean(0), X[m & ~y].mean(0)
        hit += (np.sum((X[i] - c1) ** 2) < np.sum((X[i] - c0) ** 2)) == y[i]
    return hit / len(y)


def clock_check(s, shift):
    st = s.NP[:, 1] + shift
    en = s.NP[:, 2] + shift
    X = np.zeros((s.ntrials, s.n))
    for k, t0 in enumerate(en):
        a, b = np.searchsorted(s.t, [t0 - 3.0, t0])
        X[k] = np.bincount(s.u[a:b], minlength=s.n)
    acc_sp = loo_centroid(X, s.left)
    fr = ((st[:, None] + np.linspace(2, 4, 9)) * pfc2.WHL_RATE).astype(int)
    fr = np.clip(fr, 0, len(s.whl) - 1)
    acc_x = loo_centroid(s.whl[fr, 0], s.left)
    return acc_sp, acc_x


print(f"{'session':8s} {'spikes true':>11s} {'spikes +30s':>11s} {'x true':>7s} {'x +30s':>7s}")
for k, s in S.items():
    a0, x0 = clock_check(s, 0.0)
    a1, x1 = clock_check(s, 30.0)
    print(f"{k:8s} {a0:11.2f} {a1:11.2f} {x0:7.2f} {x1:7.2f}")

## unit-count

> A total of 633 well-isolated units were recorded from anterior cingulate and dorsal prelimbic
> cortex in four rats.

The criterion: the number of distinct sorted PFC units over all working memory task recordings,
counted per recording day. The cells of each session are those of its `CellsInfo` (for EE188, the
117 of `EE188_example.mat`; the 105 of `EE188_Behavior_PFC.mat` are a variant). The data set holds
three of the paper's four rats. The files also list every cluster of the day's sorting
(`spikeph`), shown as a variant.

In [ ]:
n_cells = sum(s.n for s in S.values())
n_alt = n_cells - S["EE188"].n + len(c105)
print(f"cells over the 13 working memory sessions: {n_cells} (with EE188 as 105 cells: {n_alt}); paper 633, "
      f"within 633 +- 63.3: {abs(n_cells - 633) <= 63.3}")
print("per rat:", {r: sum(s.n for s in S.values() if s.rat == r) for r in ("EE", "FF", "GG")})
n_ph = sum((p105 if k == "EE188" else pfc2._mat(raw[k + "_Behavior_PFC.mat"]))["spikeph"].shape[0] for k in S)
print(f"all clusters listed in spikeph: {n_ph}")

**Result: reproduced.** The 13 working memory sessions of three rats hold 627 cells (615 with
EE188 counted as the 105 cells of its `Behavior_PFC` file), within 10% of the paper's 633 over four
rats; the files list 1,495 clusters in all.

## task-performance

> The rats performed the task with a mean of 91.9% correct choices during the recording sessions.

Blocked (`data_missing`): the trial table holds correct trials only (data description, page 2:
"Only correct trials are listed"), and the position file numbers the same trials, so the error
trials, and the performance, are not in the data set.

## Firing rates along the maze

Per cell, the per-trial PSTH: the normalised positions (`spikeX`) of the spikes between the trial
start (end of the nose-poke) and the trial end, as the data description defines them (page 2,
`SessionNP`; the spikes' own trial labels also cover the nose-poke, where the rat stays at
position 0 for up to 30 s), smoothed with the
Gaussian kernel of bandwidth 0.05 (Methods), so that a condition's PSTH is the mean over its
trials. Left and right PSTHs are compared by the Methods' permutation test with global bands
(1,000 permutations, level 0.05, Figure 1c legend). Only correct trials are in the data set, so
the claims file's reading T-a (correct trials only) is the one run; T-b (all trials) cannot be.

In [ ]:
def session_psths(s):
    return np.array([pfc2.smooth(s.trial_hist(i)) for i in range(s.n)])


H = {k: session_psths(s) for k, s in S.items()}
LR = {}
rng = np.random.default_rng(101)
t0 = time.time()
for k, s in S.items():
    LR[k] = [pfc2.lr_test(H[k][i], s.left, rng) for i in range(s.n)]
print("left-right permutation tests:", sum(len(v) for v in LR.values()), "cells in", round(time.time() - t0), "s")
G = pfc2.GRID


def selective(res, lo, hi, reading):
    """(a) any part of a significant segment in [lo, hi]; (b) the global band broken in [lo, hi]."""
    D0, point, glob, gp = res
    inr = (G >= lo) & (G <= hi)
    if reading == "a":
        return any(inr[a:b + 1].any() for a, b in pfc2.segments(point, glob))
    return bool((glob & inr).any())


# null check: the same test with the trial labels shuffled once per cell
rng0 = np.random.default_rng(102)
null = [pfc2.lr_test(H[k][i], rng0.permutation(s.left), rng0)[3] <= 0.05 for k, s in S.items() for i in range(s.n)]
print(f"false positive rate with shuffled labels (global, anywhere): {np.mean(null):.3f} of {len(null)} cells (level 0.05)")

### fig2c-synchrony-by-layer

> About 10% of layer 2/3 neurons and 20% of layer 5 neurons fire at least one spike in a given
> 100 ms window, and this fraction is roughly constant across the maze.

Windows of 100 ms tile each trial from its start (end of the nose-poke) to its end (trial table);
a window's position is the position file's at its centre. Reading (a) the two Figure 2a sessions:
only the layer 2/3 session (EE188) is in the data set; reading (b) all sessions of each layer,
the windows pooled.

In [ ]:
def sync_windows(s, width=0.1):
    fr, pos = [], []
    for t0, t1 in s.NP[:, 1:3]:
        edges = np.arange(t0, t1 - width + 1e-9, width)
        for a in edges:
            i, j = np.searchsorted(s.t, [a, a + width])
            fr.append(len(np.unique(s.u[i:j])) / s.n)
            f = min(int((a + width / 2) * pfc2.WHL_RATE), len(s.whl) - 1)
            pos.append(s.whl[f, 6] - 1 if s.whl[f, 6] > 0 else np.nan)
    return np.array(fr), np.array(pos)


SYNC = {k: sync_windows(s) for k, s in S.items()}


def sync_summary(keys):
    fr = np.concatenate([SYNC[k][0] for k in keys])
    pos = np.concatenate([SYNC[k][1] for k in keys])
    ok = ~np.isnan(pos)
    tenth = [fr[ok & (pos >= a / 10) & (pos < (a + 1) / 10 + (a == 9))].mean() for a in range(10)]
    return 100 * fr.mean(), 100 * np.array(tenth), len(fr), np.mean(~ok)


for k in S:
    m, tn, nw, miss = sync_summary([k])
    print(f"{k} ({S[k].layer}, {S[k].n} cells): {m:5.1f}% over {nw} windows; per tenth {np.round(tn, 1)}")
print()
for label, keys in [("(a) EE188, layer 2/3 Figure 2a session", ["EE188"]), ("(b) all layer 2/3 sessions", L23),
                    ("(b) all layer 5 sessions", L5)]:
    m, tn, nw, miss = sync_summary(keys)
    within = np.all(np.abs(tn - m) <= 0.5 * m)
    print(f"{label}: mean {m:.1f}% of cells active per 100 ms window ({nw} windows, {100 * miss:.1f}% without a position); "
          f"tenths {tn.min():.1f} to {tn.max():.1f}%, all within 50% of the mean: {within}")

**Result: partially reproduced.** Pooling all sessions of each layer (reading b), 20.3% of layer 5
cells fire in a 100 ms window (within 20 +- 2) but 12.6% of layer 2/3 cells (outside 10 +- 1); in
the Figure 2a layer 2/3 session (EE188, reading a) 11.8%. The fraction is constant along the maze:
every tenth is within 50% of the mean (layer 2/3 11.2 to 14.5%, layer 5 19.0 to 20.8%). The layer 5
Figure 2a session is not in the data set.

### fig2-population-uniform

> Individual neurons fire transiently at specific locations, but the population as a whole fires
> relatively uniformly over the whole maze.

The peak position of each cell is that of the larger of its left and right PSTH peaks (Figure 2a
orders cells by peak position in each condition). Only the layer 2/3 Figure 2a session (EE188) is
in the data set; the layer 5 sessions of rat FF are shown as a variant.

In [ ]:
def uniformity(k):
    s = S[k]
    PL = H[k][:, s.left].mean(axis=1)
    PR = H[k][:, ~s.left].mean(axis=1)
    peak = np.where(PL.max(1) >= PR.max(1), G[PL.argmax(1)], G[PR.argmax(1)])
    counts = np.histogram(peak, bins=np.linspace(0, 1, 11))[0]
    share = counts / (s.n / 10)
    chi = stats.chisquare(counts)
    pop = H[k].mean(axis=(0, 1))
    tenth = np.array([pop[(G >= a / 10) & (G < (a + 1) / 10)].mean() for a in range(10)])
    rel = tenth / pop.mean()
    ok = (share.max() <= 2) & (share.min() >= 0.5) & np.all(np.abs(rel - 1) <= 0.5)
    print(f"{k} ({s.layer}, {s.n} cells): peaks per tenth {counts.tolist()}, share of expected "
          f"{share.min():.2f} to {share.max():.2f}; chi-square against uniform {chi.statistic:.1f}, p = {chi.pvalue:.3f}; "
          f"population mean rate per tenth / overall mean {rel.min():.2f} to {rel.max():.2f}; criterion met: {ok}")
    return peak, PL, PR


peak188, PL188, PR188 = uniformity("EE188")
for k in L5:
    uniformity(k)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
o = np.argsort(peak188)
nrm = np.maximum(PL188.max(1), PR188.max(1))[:, None]
ax[0].imshow(PL188[o] / nrm, aspect="auto", extent=[0, 1, len(o), 0], vmin=0, vmax=1)
ax[1].imshow(PR188[o] / nrm, aspect="auto", extent=[0, 1, len(o), 0], vmin=0, vmax=1)
ax[0].set_title("EE188 left, rate / cell's peak (0 to 1)")
ax[1].set_title("EE188 right")
for a in ax:
    a.set_xlabel("normalised position")
plt.show()

**Result: not reproduced.** In EE188 the cells' peak positions are not spread over the maze: 17 and
61 of 117 peak in the first and last tenths and some tenths hold none (chi-square p < 0.001), and
the population mean rate per tenth ranges from 0.57 to 2.33 times its mean. The layer 5 sessions of
rat FF show the same (variant).

### lifetime-width

> Most neurons are active over a similar stretch of the maze, 0.27 (62 cm) +/- 0.17 (39 cm) of
> normalised length (mean +/- s.d.), measured between the 50% boundaries of the peak rate.

Readings of the claims file: PSTH (a) all trials, (b) left and right as two entries, (c) the
condition with the higher peak; width (i) the contiguous run around the peak above 50%, (ii) the
total length above 50%; cells (x) the Figure 2a sessions (EE188 only here), (y) all cells.

In [ ]:
def widths(P, how):
    out = []
    for p in P:
        above = p >= 0.5 * p.max()
        if how == "ii":
            out.append(above.mean())
        else:
            k = p.argmax()
            a = b = k
            while a > 0 and above[a - 1]:
                a -= 1
            while b < len(p) - 1 and above[b + 1]:
                b += 1
            out.append((b - a + 1) / len(p))
    return np.array(out)


def psth_sets(keys, which):
    P = []
    for k in keys:
        s = S[k]
        PL = H[k][:, s.left].mean(1)
        PR = H[k][:, ~s.left].mean(1)
        if which == "a":
            P.append(H[k].mean(1))
        elif which == "b":
            P += [PL, PR]
        else:
            P.append(np.where((PL.max(1) >= PR.max(1))[:, None], PL, PR))
    P = np.concatenate(P)
    return P[P.max(1) > 0]


WIDTH = {}
for cells, keys in [("x", ["EE188"]), ("y", list(S))]:
    for which in "abc":
        for how in ("i", "ii"):
            w = widths(psth_sets(keys, which), how)
            WIDTH[(which, how, cells)] = (w.mean(), w.std(ddof=1))
            ok = abs(w.mean() - 0.27) <= 0.17 and abs(w.std(ddof=1) - 0.17) <= 0.017
            print(f"PSTH {which}, width {how:2s}, cells {cells}: n = {len(w):4d}, mean {w.mean():.3f}, s.d. "
                  f"{w.std(ddof=1):.3f}, range {w.min():.3f} to {w.max():.3f}; both within tolerance: {ok}")
w = widths(psth_sets(list(S), "c"), "i")
for r in ("EE", "FF", "GG"):
    wr = widths(psth_sets([k for k in S if S[k].rat == r], "c"), "i")
    print(f"rat {r}, reading c/i: mean {wr.mean():.3f}, s.d. {wr.std(ddof=1):.3f}, n = {len(wr)}")

**Result: reproduced.** Under the PSTH over all trials with the contiguous width (readings a, i)
over all 627 cells (y) the width is 0.209 +- 0.154, within tolerance of 0.27 +- 0.17; three other
combinations also match (a, ii, x; b, ii, y; c, ii, y); in the other eight the mean is within
tolerance and the s.d. (0.097 to 0.188) is not.

### trajectory-divergence

> Left and right running paths overlap in the stem and diverge significantly from position 0.30;
> head directions diverge from 0.24.

From the position file of the Figure 2a layer 2/3 session (EE188), per trial the mean of each
variable in 100 bins of normalised position (empty bins filled by linear interpolation along the
trial), then the permutation test of the shared method at global level 0.01 on the left minus
right difference. The path is (a) the x coordinate of the LED midpoint or (b) its distance to
the mean path of all trials; head direction is the angle from LED 1 to LED 2, compared through
the circular difference of the condition means. The binning, and the absence of smoothing, are
standard choices; the smoothed version (kernel 0.05) is a variant. All sessions are a variant.

In [ ]:
NB = 100
GB = (np.arange(NB) + 0.5) / NB


def trial_paths(s):
    w = s.whl
    out = []
    for r, (t0, t1) in enumerate(s.NP[:, 1:3]):
        f = np.arange(int(t0 * pfc2.WHL_RATE), min(int(t1 * pfc2.WHL_RATE) + 1, len(w)))
        f = f[(w[f, 6] > 0) & (w[f, 0] > 0)]
        d = w[f, 6] - 1
        b = np.minimum((d * NB).astype(int), NB - 1)
        two = w[f, 2] > 0
        # LED midpoint where both LEDs are tracked, LED 1 alone otherwise
        mx = np.where(two, (w[f, 0] + w[f, 2]) / 2, w[f, 0])
        my = np.where(two, (w[f, 1] + w[f, 3]) / 2, w[f, 1])
        hd = np.where(two, np.arctan2(w[f, 3] - w[f, 1], w[f, 2] - w[f, 0]), np.nan)
        vals = []
        for v in (mx, my, np.cos(hd), np.sin(hd)):
            okv = ~np.isnan(v)
            sm = np.bincount(b[okv], v[okv], NB)
            n = np.bincount(b[okv], minlength=NB)
            m = np.where(n > 0, sm / np.maximum(n, 1), np.nan)
            ok = ~np.isnan(m)
            m = np.interp(np.arange(NB), np.flatnonzero(ok), m[ok]) if ok.sum() > 1 else np.full(NB, np.nan)
            vals.append(m)
        out.append(vals)
    return np.array(out)  # trials x 4 x NB


def perm_stat(X, labels, rng, M, stat, alpha):
    labels = np.asarray(labels, bool)
    D = np.empty((M + 1, X.shape[-1]))
    for j in range(M + 1):
        lab = labels if j == 0 else rng.permutation(labels)
        D[j] = stat(X, lab)
    point, glob, gp = pfc2.global_test(D, alpha, "two")
    return D[0], point, glob, gp


def stat_x(X, lab):
    return X[lab, 0].mean(0) - X[~lab, 0].mean(0)


def stat_dist(X, lab):
    mx, my = X[:, 0].mean(0), X[:, 1].mean(0)
    dist = np.hypot(X[:, 0] - mx, X[:, 1] - my)
    return dist[lab].mean(0) - dist[~lab].mean(0)


def stat_hd(X, lab):
    aL = np.arctan2(X[lab, 3].mean(0), X[lab, 2].mean(0))
    aR = np.arctan2(X[~lab, 3].mean(0), X[~lab, 2].mean(0))
    return np.angle(np.exp(1j * (aL - aR)))


def first_sig(res):
    D0, point, glob, gp = res
    if not glob.any():
        return np.nan, np.nan
    g = np.flatnonzero(glob)[0]
    a = g
    while a > 0 and point[a - 1]:
        a -= 1
    return GB[g], GB[a]


def smooth100(X):
    from scipy.ndimage import gaussian_filter1d
    return gaussian_filter1d(X, 0.05 * NB, axis=-1, mode="nearest")


rng = np.random.default_rng(103)
TRAJ = {}
for k in ["EE188"] + [k for k in S if k != "EE188"]:
    s = S[k]
    X = trial_paths(s)
    good = ~np.isnan(X[:, :2]).any(axis=(1, 2))
    X, lab = X[good], s.left[good]
    hd_ok = not np.isnan(X[:, 2:]).any()
    row = []
    for name, stat in [("path a (x)", stat_x), ("path b (distance to mean path)", stat_dist), ("head direction", stat_hd)]:
        if name == "head direction" and not hd_ok:
            row.append("head direction: LED 2 not tracked")
            continue
        for sm in (False, True):
            r = perm_stat(smooth100(X) if sm else X, lab, rng, 1000, stat, 0.01)
            g, a = first_sig(r)
            TRAJ[(k, name, sm)] = (g, a)
            row.append(f"{name}{' smoothed' if sm else ''}: first global {g:.2f} (segment from {a:.2f})")
    print(f"{k} ({good.sum()} of {len(good)} trials with positions): " + "; ".join(row))
    if k == "EE188":
        print()
for name in ("path a (x)", "path b (distance to mean path)"):
    g = TRAJ[("EE188", name, False)][0]
    h = TRAJ[("EE188", "head direction", False)][0]
    print(f"EE188 {name}: path {g:.2f} (0.30 +- 0.03: {abs(g - 0.30) <= 0.03}), head direction {h:.2f} "
          f"(0.24 +- 0.024: {abs(h - 0.24) <= 0.024})")

**Result: not reproduced.** In EE188 the left and right paths first differ significantly (global
level 0.01) at 0.47 (x coordinate) or 0.43 (distance to the mean path), against 0.30, and head
directions differ from 0.01, against 0.24. Smoothing gives 0.40 to 0.41 for the path. Other sessions
give values from 0.01 to 0.53.

### side-arm-selectivity-by-layer

> In the side arms (positions 0.5 to 1) the fraction of neurons firing differently on left and
> right trials is almost twice as large in layer 5 (40%) as in layer 2/3 (22%).

All cells of each layer, pooled over rats and sessions (Figure 2b legend). Readings: a cell counts
when (a) any part of a significant segment lies in 0.5 to 1, or (b) the global band is broken
within 0.5 to 1. The null rate of the test is printed above (labels shuffled).

In [ ]:
def layer_fraction(keys, lo, hi, reading):
    v = [selective(LR[k][i], lo, hi, reading) for k in keys for i in range(S[k].n)]
    return 100 * np.mean(v), int(np.sum(v)), len(v)


for reading in "ab":
    f5 = layer_fraction(L5, 0.5, 1.0, reading)
    f23 = layer_fraction(L23, 0.5, 1.0, reading)
    print(f"reading {reading}: layer 5 {f5[0]:.1f}% ({f5[1]}/{f5[2]}), within 40 +- 4: {abs(f5[0] - 40) <= 4}; "
          f"layer 2/3 {f23[0]:.1f}% ({f23[1]}/{f23[2]}), within 22 +- 2.2: {abs(f23[0] - 22) <= 2.2}")
print("per session, reading a:")
for k in S:
    f = layer_fraction([k], 0.5, 1.0, "a")
    print(f"  {k} ({S[k].layer}): {f[0]:.1f}% ({f[1]}/{f[2]})")
print("per rat, reading a:", {r: round(layer_fraction([k for k in S if S[k].rat == r], 0.5, 1.0, 'a')[0], 1) for r in ("EE", "FF", "GG")})

**Result: not reproduced.** In the side arms 62.3% of layer 5 cells (91/146) and 54.3% of layer 2/3
cells (261/481) fire differently on left and right trials (reading a; reading b 61.6% and 51.8%),
against 40% and 22%; the layer 5 fraction is larger, but not twice. With shuffled labels the test
flags 5.1% of cells.

### central-arm-selectivity

> A smaller fraction of neurons, 16% in layer 2/3 and 18% in layer 5, already fire differently on
> left and right trials in the central arm (positions 0 to 0.3).

In [ ]:
for reading in "ab":
    f23 = layer_fraction(L23, 0.0, 0.3, reading)
    f5 = layer_fraction(L5, 0.0, 0.3, reading)
    print(f"reading {reading}: layer 2/3 {f23[0]:.1f}% ({f23[1]}/{f23[2]}), within 16 +- 1.6: {abs(f23[0] - 16) <= 1.6}; "
          f"layer 5 {f5[0]:.1f}% ({f5[1]}/{f5[2]}), within 18 +- 1.8: {abs(f5[0] - 18) <= 1.8}")
for k in S:
    f = layer_fraction([k], 0.0, 0.3, "a")
    print(f"  {k} ({S[k].layer}): {f[0]:.1f}% ({f[1]}/{f[2]})")
print("per rat, reading a:", {r: round(layer_fraction([k for k in S if S[k].rat == r], 0.0, 0.3, 'a')[0], 1) for r in ("EE", "FF", "GG")})

**Result: not reproduced.** In the central arm 31.4% of layer 2/3 cells and 26.7% of layer 5 cells
are selective (reading a; reading b 28.7% and 22.6%), against 16% and 18%.

### odor-selective-fraction

> About a quarter of neurons (26% in one layer 2/3 session) fire at significantly different rates
> during nose-poking for the two odors.

Rate during each trial's nose-poke, from the first nose-poke to the end of the last (trial table,
columns 1 and 2, which the data description defines from the photoelectric sensor), left (one
odor) against right (the other) trials, t-test at p < 0.05. Session EE188 (the Figure 2a layer
2/3 session); readings (a) Student, (b) Welch. Other sessions are a variant.

In [ ]:
def nosepoke_rates(s):
    R = np.zeros((s.ntrials, s.n))
    for r, (a, b) in enumerate(s.NP[:, :2]):
        i, j = np.searchsorted(s.t, [a, b])
        R[r] = np.bincount(s.u[i:j], minlength=s.n) / (b - a)
    return R


ODOR = {}
for k, s in S.items():
    R = nosepoke_rates(s)
    for reading, eq in (("a", True), ("b", False)):
        with np.errstate(invalid="ignore", divide="ignore"):
            p = stats.ttest_ind(R[s.left], R[~s.left], equal_var=eq).pvalue
        ODOR[(k, reading)] = (p < 0.05, R[s.left].mean(0) - R[~s.left].mean(0), p)
    if k == "EE188":
        for reading in "ab":
            sig = ODOR[(k, reading)][0]
            print(f"EE188 reading {reading}: {100 * sig.mean():.1f}% ({sig.sum()}/{s.n}) odor selective; "
                  f"within 26 +- 2.6: {abs(100 * sig.mean() - 26) <= 2.6}")
        print(f"nose-poke durations EE188: median {np.median(s.NP[:, 1] - s.NP[:, 0]):.1f} s, range "
              f"{np.min(s.NP[:, 1] - s.NP[:, 0]):.1f} to {np.max(s.NP[:, 1] - s.NP[:, 0]):.1f} s")
print("other sessions, reading a:", {k: f"{100 * ODOR[(k, 'a')][0].mean():.0f}%" for k in S if k != "EE188"})
allsig = np.concatenate([ODOR[(k, "a")][0] for k in S])
print(f"all sessions, reading a: {100 * allsig.mean():.1f}% of {len(allsig)} cells")

**Result: not reproduced.** In EE188, 45.3% of cells (53/117, Student's t-test; Welch 42.7%) fire at
different rates during the nose-poke on left and right trials, against 26%; over all sessions 32.4%.

### odor-unrelated-to-stem

> Neurons' odor preference during nose-poking is unrelated to their left-right differences in the
> stem and side arms.

Session EE188. Readings (a) Fisher exact test of odor-selective (reading a of the odor claim)
against selective in the region (reading a of the arm claims); (b) Spearman correlation across
cells of the nose-poke rate difference (left minus right) and the mean left minus right PSTH in
the region. The claims file asks for a correlation at p < 0.05 without naming the coefficient;
Pearson is shown as a variant.

In [ ]:
s = S["EE188"]
od_sig, od_diff, _ = ODOR[("EE188", "a")]
assoc = False
for name, lo, hi in (("stem 0-0.3", 0.0, 0.3), ("side arms 0.5-1", 0.5, 1.0)):
    reg = np.array([selective(LR["EE188"][i], lo, hi, "a") for i in range(s.n)])
    tab = np.array([[np.sum(od_sig & reg), np.sum(od_sig & ~reg)], [np.sum(~od_sig & reg), np.sum(~od_sig & ~reg)]])
    odds, pf = stats.fisher_exact(tab)
    inr = (G >= lo) & (G <= hi)
    dreg = np.array([LR["EE188"][i][0][inr].mean() for i in range(s.n)])
    rs = stats.spearmanr(od_diff, dreg)
    rp = stats.pearsonr(od_diff, dreg)
    same_sign = (odds > 1 and pf < 0.05) or (rs.statistic > 0 and rs.pvalue < 0.05)
    assoc |= same_sign
    print(f"{name}: table [[odor & region, odor only], [region only, neither]] = {tab.tolist()}, Fisher odds "
          f"{odds:.2f}, p = {pf:.3f}; Spearman r = {rs.statistic:.3f}, p = {rs.pvalue:.3f}; Pearson r = "
          f"{rp.statistic:.3f}, p = {rp.pvalue:.3f}")
print("an association with the same sign of preference at p < 0.05 in either region:", assoc)

**Result: reproduced.** In EE188 neither reading finds an association with the same sign: Fisher p =
0.191 (stem) and 0.267 (side arms); Spearman r = -0.184 (p = 0.047) in the stem, which is of
opposite sign, and 0.134 (p = 0.150) in the side arms.

### retrospective-coding-rare

> Only a small fraction of neurons fire differently according to the reward location of the
> previous trial.

Trials are labelled by the side of the trial before them in the trial table; the first trial of
each session has no label and is left out. The table lists correct trials only, so where an error
trial was left out, the trial before is the previous correct trial (a deviation). The current-trial
fraction is computed on the same trials. Readings (a) the Figure 2a session (EE188, layer 2/3; the
layer 5 session is not in the data set), (b) all sessions of each layer.

In [ ]:
rng = np.random.default_rng(104)
RETRO = {}
for k, s in S.items():
    prev = s.left[:-1]
    cur = s.left[1:]
    RETRO[k] = [(pfc2.lr_test(H[k][i][1:], prev, rng)[3] <= 0.05, pfc2.lr_test(H[k][i][1:], cur, rng)[3] <= 0.05)
                for i in range(s.n)]
    same = np.mean(prev == cur)
    if k == "EE188":
        print(f"EE188: previous side equals current side in {100 * same:.0f}% of trials")
for label, keys in (("(a) EE188", ["EE188"]), ("(b) layer 2/3", L23), ("(b) layer 5", L5)):
    v = np.array([x for k in keys for x in RETRO[k]])
    fp, fc = 100 * v[:, 0].mean(), 100 * v[:, 1].mean()
    print(f"{label}: previous-trial labels {fp:.1f}% ({v[:, 0].sum()}/{len(v)}), current-trial labels {fc:.1f}% "
          f"({v[:, 1].sum()}); previous below half of current: {fp < fc / 2}")
for k in S:
    v = np.array(RETRO[k])
    print(f"  {k}: previous {v[:, 0].sum()}/{len(v)}, current {v[:, 1].sum()}/{len(v)}, trials {S[k].ntrials}, "
          f"previous = current in {100 * np.mean(S[k].left[:-1] == S[k].left[1:]):.0f}%")

**Result: reproduced.** Labelled by the previous trial's side, 4.3% of EE188 cells (5/117) are
selective against 77.8% for the current side; over all sessions 7.3% (layer 2/3) and 8.9% (layer 5)
against 66.9% and 65.8%.

### speed-not-correlated

> Firing rates show no net positive relationship with running speed; the distribution of
> rate-speed correlations is symmetric around zero.

Supplementary Figure 3C: within trials (trial start to end), 0.5 s bins; the speed of each bin is
the LED midpoint's (LED 1 where LED 2 is not tracked) path length over the bin; the rate is the
spike count over 0.5 s; one Pearson correlation per cell over all bins of its session. Two-sided
Wilcoxon signed-rank test of the correlations against 0, over all cells and within the putative
pyramidal cells and interneurons (cell types from the connection claims below, and from the
files' `CellsInfo` as a variant).

In [ ]:
def speed_corr(s):
    rows, cnt = [], []
    w = s.whl
    for t0, t1 in s.NP[:, 1:3]:
        for a in np.arange(t0, t1 - 0.5 + 1e-9, 0.5):
            f = np.arange(int(a * pfc2.WHL_RATE), int((a + 0.5) * pfc2.WHL_RATE) + 1)
            f = f[f < len(w)]
            ok = w[f, 0] > 0
            if ok.sum() < 3:
                continue
            two = w[f, 2] > 0
            x = np.where(two, (w[f, 0] + w[f, 2]) / 2, w[f, 0])[ok]
            y = np.where(two, (w[f, 1] + w[f, 3]) / 2, w[f, 1])[ok]
            rows.append(np.sum(np.hypot(np.diff(x), np.diff(y))) / (len(x) - 1) * pfc2.WHL_RATE)
            i, j = np.searchsorted(s.t, [a, a + 0.5])
            cnt.append(np.bincount(s.u[i:j], minlength=s.n))
    sp = np.array(rows)
    C = np.array(cnt, float)
    r = np.array([np.corrcoef(sp, C[:, i])[0, 1] if C[:, i].std() > 0 else np.nan for i in range(s.n)])
    return r, len(sp)


SPEED = {}
for k, s in S.items():
    SPEED[k], nb = speed_corr(s)
r_all = np.concatenate([SPEED[k] for k in S])
r_all = r_all[~np.isnan(r_all)]
w_all = stats.wilcoxon(r_all)
print(f"all cells: n = {len(r_all)}, median r = {np.median(r_all):.3f}, mean {r_all.mean():.3f}, "
      f"fraction positive {np.mean(r_all > 0):.2f}, Wilcoxon p = {w_all.pvalue:.2g}; "
      f"median significantly above 0: {np.median(r_all) > 0 and w_all.pvalue < 0.05}")
print(f"range of r: {r_all.min():.2f} to {r_all.max():.2f}")
for k in S:
    r = SPEED[k][~np.isnan(SPEED[k])]
    print(f"  {k}: median r = {np.median(r):.3f}, n = {len(r)}")
for r_ in ("EE", "FF", "GG"):
    r = np.concatenate([SPEED[k] for k in S if S[k].rat == r_])
    r = r[~np.isnan(r)]
    print(f"  rat {r_}: median r = {np.median(r):.3f}, n = {len(r)}, Wilcoxon p = {stats.wilcoxon(r).pvalue:.2g}")
ty = np.concatenate([S[k].celltype_file for k in S])
rr = np.concatenate([SPEED[k] for k in S])
for t, name in ((1, "pyramidal"), (2, "interneuron")):
    v = rr[(ty == t) & ~np.isnan(rr)]
    print(f"  CellsInfo {name}: median r = {np.median(v):.3f}, n = {len(v)}, Wilcoxon p = {stats.wilcoxon(v).pvalue:.2g}")

**Result: reproduced.** Over 627 cells the median rate-speed correlation is -0.018 (45% positive),
not above 0; the Wilcoxon test is significant (p = 0.00036) in the negative direction, so the
distribution is shifted slightly below zero, not symmetric.

### population-vector-width

> Population vector correlations fall off over similar distances throughout the maze, 0.18 +/-
> 0.087 of normalised length in layer 2/3 and 0.27 +/- 0.13 in layer 5.

Population vectors of the cells' PSTHs at the 200 positions, Pearson correlation between
positions; width at a position, the contiguous length around the diagonal over which the
correlation stays above half its diagonal value (1); mean +/- s.d. over positions. Readings (a)
left and right PSTHs concatenated in each vector, (b) all trials. Sessions: the Figure 2a layer
2/3 session (EE188; the layer 5 one is not in the data set), and all sessions of each layer
pooled into one vector (the variant the claims file names), which gives the only layer 5 value.
Whether each cell's PSTH is normalised before the correlation is not said; raw rates, rates over
the cell's peak (as Figure 2a displays them) and z-scored rates are all run.

In [ ]:
def pv_width(keys, which, norm):
    P = []
    for k in keys:
        s = S[k]
        if which == "a":
            P.append(np.concatenate([H[k][:, s.left].mean(1), H[k][:, ~s.left].mean(1)], axis=0))
        else:
            P.append(H[k].mean(1))
    P = np.concatenate(P)
    P = P[P.std(1) > 0]
    if norm == "peak":
        P = P / P.max(1, keepdims=True)
    elif norm == "z":
        P = (P - P.mean(1, keepdims=True)) / P.std(1, keepdims=True)
    C = np.corrcoef(P.T)
    wd = []
    for x in range(len(C)):
        above = C[x] > 0.5
        a = b = x
        while a > 0 and above[a - 1]:
            a -= 1
        while b < len(C) - 1 and above[b + 1]:
            b += 1
        wd.append((b - a + 1) / len(C))
    return np.array(wd), C


PV = {}
for norm in ("raw", "peak", "z"):
  for label, keys in (("EE188 (L2/3)", ["EE188"]), ("all L2/3", L23), ("all L5", L5)):
    for which in "ab":
        wd, C = pv_width(keys, which, norm)
        PV[(label, which, norm)] = (wd.mean(), wd.std(ddof=1))
        print(f"{norm:4s} {label}, reading {which}: width {wd.mean():.3f} +- {wd.std(ddof=1):.3f} (range {wd.min():.3f} to "
              f"{wd.max():.3f}); correlations {C.min():.2f} to {C.max():.2f}")
for norm in ("raw", "peak", "z"):
  for which in "ab":
    l23 = PV[("EE188 (L2/3)", which, norm)][0]
    l23p = PV[("all L2/3", which, norm)][0]
    l5 = PV[("all L5", which, norm)][0]
    print(f"{norm:4s} reading {which}: L2/3 EE188 {l23:.3f} (0.18 +- 0.087: {abs(l23 - 0.18) <= 0.087}), L2/3 pooled {l23p:.3f} "
          f"({abs(l23p - 0.18) <= 0.087}), L5 pooled {l5:.3f} (0.27 +- 0.13: {abs(l5 - 0.27) <= 0.13})")

**Result: reproduced, resting on an assumption.** With raw rates the population vectors correlate
above 0.5 everywhere, so the width is the whole maze (0.96 to 1.00) and the claim fails; with each
cell's PSTH normalised (z-scored, or divided by its peak), EE188 gives 0.182 (z, reading a) and the
pooled layer 5 sessions 0.204, within 0.18 +- 0.087 and 0.27 +- 0.13; all normalised readings match
except the pooled layer 2/3 peak-normalised reading b (0.268). The layer 5 value comes from the
pooled sessions, as the Figure 2a layer 5 session is not in the data set.

## Monosynaptic connections

For every session, the cross-correlogram of every ordered pair of cells over the whole session
(Figure 3b includes the intertrial intervals), against 1,000 surrogates in which every spike of
every cell is jittered uniformly on [-5, +5] ms (Methods). Connection: the counts break the global
99% band anywhere in [1, 4] ms, above (excitatory, a peak) or below (inhibitory, a trough). Readings
of the claims file: bin width B-a 1 ms or B-b 0.5 ms; width rule W-a the global band test alone,
W-b also requiring the run of bins beyond the pointwise 99% band around the extreme to span 2 ms
or less. The global band is the 99th percentile of the surrogates' maxima over [-5, +5] ms (upper)
and the 1st percentile of their minima (lower): the Methods give the level (99%) and the source
(the maximum and minimum of each surrogate) but not how the level is split between the two
sides. 1% on each side (G-a) is the reading used for every claim that follows from the
connections; 0.5% on each side (G-b) is run for the counts of connection-counts and
cell-type-fractions to show how much they depend on it. Both are checked against the connections
the authors list for session EE188 (`pairs_all` of `EE188_example.mat`).

In [ ]:
CONN = {}
t0 = time.time()
for bw, tag in ((0.001, "B-a"), (0.0005, "B-b")):
    for k, s in S.items():
        rng = np.random.default_rng(200 + list(S).index(k) + (0 if tag == "B-a" else 50))
        obs, sur = pfc2.ccg_jitter(s.t, s.u, s.n, rng, bw, nsur=1000)
        c = pfc2.detect_connections(obs, sur, bw)
        CONN[(k, tag)] = {"exc": c["exc"], "inh": c["inh"], "width": c["width"]}
        c2 = pfc2.detect_connections(obs, sur, bw, gside=0.005)
        CONN[(k, tag, "G-b")] = {"exc": c2["exc"], "inh": c2["inh"], "width": c2["width"]}
        del obs, sur, c, c2
print("jitter CCGs of", sum(s.n * (s.n - 1) for s in S.values()), "ordered pairs, two bin widths, in",
      round(time.time() - t0), "s")


def conns(k, B, W, G="G-a"):
    c = CONN[(k, B)] if G == "G-a" else CONN[(k, B, G)]
    E, I = c["exc"].copy(), c["inh"].copy()
    if W == "W-b":
        E &= c["width"] <= 2
        I &= c["width"] <= 2
    return E, I


COMBOS = [(b, w) for b in ("B-a", "B-b") for w in ("W-a", "W-b")]

# check against the authors' list for EE188
s = S["EE188"]
idx = {c: i for i, c in enumerate(s.cells)}
PA = np.zeros((s.n, s.n), bool)
for a, b in EX["pairs_all"].astype(int):
    PA[idx[a], idx[b]] = True
pre_t = s.celltype_file[PA.any(1)]
post_t = s.celltype_file[PA.any(0)]
print(f"EE188 pairs_all: {PA.sum()} pairs; presynaptic cells {PA.any(1).sum()} (types {np.bincount(pre_t, minlength=4)[1:]}), "
      f"targets {PA.any(0).sum()} (types {np.bincount(post_t, minlength=4)[1:]}) [types: pyramidal, interneuron, unknown]")
for gs in ("G-a", "G-b"):
    for B, W in COMBOS:
        E, I = conns("EE188", B, W, gs)
        print(f"  {gs} {B} {W}: excitatory {E.sum()}, inhibitory {I.sum()}; of the {PA.sum()} pairs_all found as "
              f"excitatory {(PA & E).sum()}, as inhibitory {(PA & I).sum()}")

### connection-counts

> Of 62,408 directed cell pairs, 495 (0.79%) show a short-latency narrow peak or trough in the
> cross-correlogram, 0.55% excitatory and 0.24% inhibitory, and 0.17% of pairs are reciprocally
> connected.

Reciprocal: unordered pairs connected in both directions (any type); denominator (a) directed
pairs, (b) unordered pairs. The data set holds three of the paper's four rats.

In [ ]:
NPAIRS = sum(s.n * (s.n - 1) for s in S.values())
NCELLS = sum(s.n for s in S.values())
print(f"directed pairs of simultaneously recorded cells: {NPAIRS} (paper 62,408; within 10%: {abs(NPAIRS - 62408) <= 6240.8})")
for gs, B, W in [(g, b, w) for g in ("G-a", "G-b") for b, w in COMBOS]:
    ne = ni = nany = nrec = 0
    for k in S:
        E, I = conns(k, B, W, gs)
        A = E | I
        ne += E.sum()
        ni += I.sum()
        nany += A.sum()
        nrec += np.triu(A & A.T, 1).sum()
    pa, pe, pi = 100 * nany / NPAIRS, 100 * ne / NPAIRS, 100 * ni / NPAIRS
    ra, rb = 100 * nrec / NPAIRS, 100 * nrec / (NPAIRS / 2)
    ok3 = abs(pa - 0.79) <= 0.079 and abs(pe - 0.55) <= 0.055 and abs(pi - 0.24) <= 0.024
    print(f"{gs} {B} {W}: any {nany} ({pa:.2f}%), excitatory {ne} ({pe:.2f}%), inhibitory {ni} ({pi:.2f}%), "
          f"reciprocal pairs {nrec} = {ra:.3f}% of directed (a), {rb:.3f}% of unordered (b); "
          f"three directed values within tolerance: {ok3}; reciprocal within 0.17 +- 0.017: (a) {abs(ra - 0.17) <= 0.017}, (b) {abs(rb - 0.17) <= 0.017}")
print("per rat (B-a W-a):")
for r in ("EE", "FF", "GG"):
    ks = [k for k in S if S[k].rat == r]
    npr = sum(S[k].n * (S[k].n - 1) for k in ks)
    ne = sum(conns(k, "B-a", "W-a")[0].sum() for k in ks)
    ni = sum(conns(k, "B-a", "W-a")[1].sum() for k in ks)
    print(f"  {r}: {npr} pairs, excitatory {100 * ne / npr:.2f}%, inhibitory {100 * ni / npr:.2f}%")

**Result: not reproduced.** The data set gives 40,240 directed pairs (three of four rats, against
62,408). Under G-a B-a W-a, 1.50% of pairs are connected, 1.09% excitatory and 0.47% inhibitory; the
strictest combination (G-a B-b W-b) gives 1.07%, 0.94% and 0.14%, against 0.79%, 0.55% and 0.24%.
With the global level split between the sides (G-b) the values are 0.83 to 1.20%, 0.73 to 0.86% and
0.11 to 0.38%, and no combination puts all three within tolerance. Reciprocal pairs are 0.06 to
0.23% of directed pairs under G-a, and 0.179% of unordered pairs under G-a B-b W-a.

### cell-type-fractions

> 32.5% of units are putative pyramidal cells and 12.5% putative interneurons; 50.7% of the
> targets of excitatory cells themselves suppress other neurons; the ratio of pyramidal cells to
> interneurons is 2.82 +/- 0.51 (s.d.).

A cell with a significant excitatory connection as reference is a putative pyramidal cell, one
with an inhibitory one a putative interneuron. A cell with both: C-a unclassified, C-b excluded
from the denominators. The ratio: mean +/- s.d. over (a) sessions, (b) rats; units without an
interneuron cannot give a ratio and are left out (counted below).

In [ ]:
def celltypes(k, B, W, C):
    E, I = conns(k, B, W)
    pyr, inn = E.any(1), I.any(1)
    both = pyr & inn
    return pyr & ~both, inn & ~both, both


TYPES = {}
for B, W in COMBOS:
    for C in ("C-a", "C-b"):
        npyr = nint = nboth = ntot = 0
        tgt_int = tgt = 0
        per_s, per_r = [], {}
        for k in S:
            pyr, inn, both = celltypes(k, B, W, C)
            TYPES[(k, B, W, C)] = (pyr, inn)
            E, I = conns(k, B, W)
            npyr += pyr.sum()
            nint += inn.sum()
            nboth += both.sum()
            ntot += S[k].n - (both.sum() if C == "C-b" else 0)
            targets = E.any(0)
            tgt += targets.sum()
            tgt_int += (targets & I.any(1)).sum()
            if inn.sum() > 0:
                per_s.append(pyr.sum() / inn.sum())
            r = S[k].rat
            per_r.setdefault(r, [0, 0])
            per_r[r][0] += pyr.sum()
            per_r[r][1] += inn.sum()
        rs = np.array(per_s)
        rr = np.array([a / b for a, b in per_r.values() if b > 0])
        fp, fi, ft = 100 * npyr / ntot, 100 * nint / ntot, 100 * tgt_int / tgt
        ok = (abs(fp - 32.5) <= 3.25, abs(fi - 12.5) <= 1.25, abs(ft - 50.7) <= 5.07,
              abs(rs.mean() - 2.82) <= 0.51, abs(rr.mean() - 2.82) <= 0.51)
        print(f"{B} {W} {C}: pyramidal {fp:.1f}% ({npyr}/{ntot}), interneurons {fi:.1f}% ({nint}), both {nboth}; "
              f"targets of excitation that are interneurons {ft:.1f}% ({tgt_int}/{tgt}); ratio per session "
              f"{rs.mean():.2f} +- {rs.std(ddof=1):.2f} (n = {len(rs)} of {len(S)}), per rat {rr.mean():.2f} +- "
              f"{rr.std(ddof=1) if len(rr) > 1 else np.nan:.2f} (n = {len(rr)}); within tolerance "
              f"[pyr, int, targets, ratio a, ratio b]: {ok}")
for B, W in COMBOS:
    npyr = nint = tgt = tgt_int = 0
    for k in S:
        E, I = conns(k, B, W, "G-b")
        pyr, inn = E.any(1) & ~I.any(1), I.any(1) & ~E.any(1)
        npyr += pyr.sum()
        nint += inn.sum()
        tgt += E.any(0).sum()
        tgt_int += (E.any(0) & I.any(1)).sum()
    print(f"G-b {B} {W} C-a: pyramidal {100 * npyr / NCELLS:.1f}%, interneurons {100 * nint / NCELLS:.1f}%, targets of "
          f"excitation that are interneurons {100 * tgt_int / max(tgt, 1):.1f}%")
ci = S["EE188"].celltype_file
print(f"EE188 CellsInfo of the example file: pyramidal {np.mean(ci == 1) * 100:.1f}%, interneuron "
      f"{np.mean(ci == 2) * 100:.1f}%, unknown {np.mean(ci == 3) * 100:.1f}% of {len(ci)}")
for B, W in COMBOS:
    pyr, inn = TYPES[("EE188", B, W, "C-a")]
    print(f"  {B} {W}: agreement with CellsInfo; our pyramidal cells typed 1/2/3 {np.bincount(ci[pyr], minlength=4)[1:]}, "
          f"our interneurons {np.bincount(ci[inn], minlength=4)[1:]}")

**Result: not reproduced.** Under every G-a combination 37.8 to 41.8% of cells are putative
pyramidal cells (32.5% in the paper) and 2.2 to 6.4% interneurons (12.5%); 18.2 to 31.8% of the
targets of excitation are interneurons (50.7%); the pyramidal to interneuron ratio is 6.5 to 18.9
per session (2.82). With G-b the pyramidal fraction is 32.2 to 33.5%, within tolerance, but
interneurons are 2.6 to 5.6%. The authors' own types for EE188 (`CellsInfo` of the example file) are
28.2% pyramidal and 12.8% interneurons.

### pyr-pyr-connections

> Monosynaptic excitation between putative pyramidal cells is rare, found in 0.12% of layer 2/3
> and 0.27% of layer 5 pairs.

Excitatory connections whose target is also a putative pyramidal cell; denominator (a) all
directed pairs of the layer, (b) directed pairs of two putative pyramidal cells.

In [ ]:
for B, W in COMBOS:
    for C in ("C-a", "C-b"):
        out = []
        for lay, keys in (("L2/3", L23), ("L5", L5)):
            npp = nall = npyrpairs = 0
            for k in keys:
                E, I = conns(k, B, W)
                pyr, inn = TYPES[(k, B, W, C)]
                npp += (E & pyr[:, None] & pyr[None, :]).sum()
                nall += S[k].n * (S[k].n - 1)
                npyrpairs += pyr.sum() * (pyr.sum() - 1)
            out.append((lay, npp, 100 * npp / nall, 100 * npp / max(npyrpairs, 1)))
        ok_a = abs(out[0][2] - 0.12) <= 0.012 and abs(out[1][2] - 0.27) <= 0.027
        ok_b = abs(out[0][3] - 0.12) <= 0.012 and abs(out[1][3] - 0.27) <= 0.027
        print(f"{B} {W} {C}: " + "; ".join(f"{l} {n} connections, {a:.3f}% of all pairs, {b:.2f}% of pyramidal pairs" for l, n, a, b in out)
              + f"; within tolerance (a) {ok_a}, (b) {ok_b}")

**Result: not reproduced.** Pyramidal to pyramidal excitation is found in 0.164 to 0.176% of layer
2/3 pairs (0.12%) and 0.191 to 0.246% of layer 5 pairs (0.27%); only the layer 5 value under B-b W-a
is within tolerance.

### fig3b-session-connections

> In the single layer 2/3 session of Figure 3b (1172 pairs), 48 connections have a putative
> pyramidal reference cell and 30 a putative interneuron; in the layer 5 session of
> Supplementary Figure 11 (about 1422 pairs) the numbers are 73 and 25.

Session EE188 (117 cells, the Figure 3 session by the data description). The layer 5 session (142
cells) is not in the data set, so its two values cannot be tested. Readings (a) directed
connections, (b) distinct reference cells; connections of a reference cell typed as both are
counted under neither in C-a.

In [ ]:
s = S["EE188"]
print(f"EE188: {s.n} cells, {s.n * (s.n - 1)} ordered pairs (paper: 1172 pairs, 13,572 matrix points)")
for B, W in COMBOS:
    E, I = conns("EE188", B, W)
    pyr, inn = TYPES[("EE188", B, W, "C-a")]
    A = E | I
    a_p, a_i = A[pyr].sum(), A[inn].sum()
    b_p, b_i = pyr.sum(), inn.sum()
    print(f"{B} {W}: (a) connections from pyramidal references {a_p}, from interneurons {a_i}; (b) reference cells "
          f"{b_p} and {b_i}; within 48 +- 4.8 and 30 +- 3: (a) {abs(a_p - 48) <= 4.8 and abs(a_i - 30) <= 3}, "
          f"(b) {abs(b_p - 48) <= 4.8 and abs(b_i - 30) <= 3}")
print(f"authors' pairs_all: {PA.sum()} connections from {PA.any(1).sum()} reference cells")

**Result: not reproduced.** In EE188, 84 to 86 connections have a pyramidal reference cell and 5 to
21 an interneuron, against 48 and 30 (55 to 56 and 2 to 11 reference cells). The authors'
`pairs_all` lists 43 connections, all of which the B-a test finds. The layer 5 session of
Supplementary Figure 11 is not in the data set.

### fig3d-hub

> A large part of the recorded neurons belong to a single interconnected circuit, 39 of 117 in the
> layer 2/3 session and 45 of 142 in the layer 5 session.

Largest connected component of the undirected graph of excitatory and inhibitory connections in
EE188. The layer 5 session is not in the data set. The authors' `pairs_all` graph is shown for
comparison.

In [ ]:
def largest_component(A):
    g = nx.Graph()
    g.add_edges_from(zip(*np.nonzero(A)))
    return max((len(c) for c in nx.connected_components(g)), default=0)


for B, W in COMBOS:
    E, I = conns("EE188", B, W)
    n = largest_component(E | I)
    print(f"{B} {W}: largest connected circuit {n} of {S['EE188'].n} cells; within 39 +- 3.9: {abs(n - 39) <= 3.9}")
print(f"authors' pairs_all graph: largest component {largest_component(PA)} cells")

**Result: not reproduced.** The largest connected circuit in EE188 holds 85 or 86 (B-a) or 64 (B-b)
of 117 cells, against 39; the authors' `pairs_all` graph has a largest component of exactly 39
cells, so Figure 3d draws a subset of the connections this test finds. The layer 5 session is not in
the data set.

### fig3f-distance

> Most connected pairs are on the same shank and connection probability falls rapidly with the
> distance between the neurons, yet connections are found up to 1,200 um apart.

Connection probability by shank separation (200 um per shank), excitatory and inhibitory
separately, with exact Clopper-Pearson 99% intervals, all sessions.

In [ ]:
def cp_interval(k, n, level=0.99):
    a = 1 - level
    lo = stats.beta.ppf(a / 2, k, n - k + 1) if k > 0 else 0.0
    hi = stats.beta.ppf(1 - a / 2, k + 1, n - k) if k < n else 1.0
    return lo, hi


for B, W in COMBOS:
    cnt = {}
    for k, s in S.items():
        E, I = conns(k, B, W)
        sep = np.abs(s.shank[:, None] - s.shank[None, :]) * 200
        off = ~np.eye(s.n, dtype=bool)
        for d in np.unique(sep[off]):
            m = (sep == d) & off
            c = cnt.setdefault(d, [0, 0, 0])
            c[0] += m.sum()
            c[1] += (E & m).sum()
            c[2] += (I & m).sum()
    ds = sorted(cnt)
    same = cnt[0][1] + cnt[0][2]
    other = sum(cnt[d][1] + cnt[d][2] for d in ds if d > 0)
    p0 = (cnt[0][1] + cnt[0][2]) / cnt[0][0]
    lo0, hi0 = cp_interval(cnt[0][1] + cnt[0][2], cnt[0][0])
    lo2, hi2 = cp_interval(cnt[200][1] + cnt[200][2], cnt[200][0])
    p2 = (cnt[200][1] + cnt[200][2]) / cnt[200][0]
    far = max(d for d in ds if cnt[d][1] + cnt[d][2] > 0)
    print(f"{B} {W}: connected pairs on the same shank {same}, on other shanks {other}; same-shank probability "
          f"{100 * p0:.2f}% [{100 * lo0:.2f}, {100 * hi0:.2f}] against 200 um {100 * p2:.2f}% [{100 * lo2:.2f}, {100 * hi2:.2f}]; "
          f"largest separation with a connection {far} um; criterion met: {same > other and lo0 > hi2 and far == 1200}")
    if (B, W) == ("B-a", "W-a"):
        for d in ds:
            e_lo, e_hi = cp_interval(cnt[d][1], cnt[d][0])
            i_lo, i_hi = cp_interval(cnt[d][2], cnt[d][0])
            print(f"    {d:5d} um: {cnt[d][0]:6d} pairs, excitatory {100 * cnt[d][1] / cnt[d][0]:.2f}% [{100 * e_lo:.2f}, "
                  f"{100 * e_hi:.2f}], inhibitory {100 * cnt[d][2] / cnt[d][0]:.2f}% [{100 * i_lo:.2f}, {100 * i_hi:.2f}]")

**Result: partially reproduced.** Most connected pairs are on the same shank (396 against 234 on
other shanks, B-a W-a) and the same-shank probability (6.98%, 99% interval 6.13 to 7.89) is far
above that at 200 um (1.44%, 1.14 to 1.78), falling with distance; but connections are found up to
1,400 um, the largest separation on the probe, not 1,200.

### nearby-pyr-int-probability

> Connection probability between nearby (under 200 um) putative pyramidal cells and interneurons
> is 1 to 2%.

Same-shank pairs of a putative pyramidal cell and a putative interneuron, all sessions. Readings
(a) pyramidal to interneuron excitatory connections over such directed pairs, (b) a connection in
either direction over unordered pairs.

In [ ]:
for B, W in COMBOS:
    for C in ("C-a", "C-b"):
        na = ka = kb = 0
        for k, s in S.items():
            E, I = conns(k, B, W)
            pyr, inn = TYPES[(k, B, W, C)]
            m = (s.shank[:, None] == s.shank[None, :]) & pyr[:, None] & inn[None, :]
            na += m.sum()
            ka += (E & m).sum()
            kb += ((E | I | E.T | I.T) & m).sum()
        pa_, pb_ = 100 * ka / max(na, 1), 100 * kb / max(na, 1)
        print(f"{B} {W} {C}: {na} same-shank pyramidal-interneuron pairs; (a) {ka} = {pa_:.1f}%, (b) {kb} = {pb_:.1f}%; "
              f"in 0.9 to 2.2%: (a) {0.9 <= pa_ <= 2.2}, (b) {0.9 <= pb_ <= 2.2}")

**Result: not reproduced.** Of same-shank pairs of a putative pyramidal cell and interneuron, 33.8
to 59.4% are connected (reading a) and 38.8 to 60.9% (reading b), against 1 to 2%. Cell types here
come from the connections themselves, so the pairs counted are mostly those that defined the types.

### connected-cells-trajectory-specific

> Most putative pyramidal cells (72.2% in layer 2/3, 84.7% in layer 5) and many interneurons
> (58.3% and 83%) fire differently on left and right trajectories.

A cell is trajectory specific when the left-right permutation test (above) finds a significant
segment anywhere. Readings (a) the identified cells of the Figure 3 sessions (EE188 for layer
2/3; the layer 5 session is not in the data set), (b) all sessions of each layer.

In [ ]:
def traj_frac(keys, B, W, C):
    out = {}
    for name, j in (("pyr", 0), ("int", 1)):
        v = np.concatenate([[LR[k][i][3] <= 0.05 for i in np.flatnonzero(TYPES[(k, B, W, C)][j])] for k in keys])
        out[name] = (100 * v.mean() if len(v) else np.nan, int(v.sum()), len(v))
    return out


for B, W in COMBOS:
    for C in ("C-a",):
        a23 = traj_frac(["EE188"], B, W, C)
        b23 = traj_frac(L23, B, W, C)
        b5 = traj_frac(L5, B, W, C)
        ok_b = (abs(b23["pyr"][0] - 72.2) <= 7.22 and abs(b5["pyr"][0] - 84.7) <= 8.47 and
                abs(b23["int"][0] - 58.3) <= 5.83 and abs(b5["int"][0] - 83) <= 8.3)
        print(f"{B} {W} {C}: (a) EE188 pyramidal {a23['pyr'][0]:.1f}% ({a23['pyr'][1]}/{a23['pyr'][2]}), interneurons "
              f"{a23['int'][0]:.1f}% ({a23['int'][1]}/{a23['int'][2]}); (b) L2/3 pyramidal {b23['pyr'][0]:.1f}% "
              f"({b23['pyr'][1]}/{b23['pyr'][2]}), interneurons {b23['int'][0]:.1f}% ({b23['int'][1]}/{b23['int'][2]}); "
              f"L5 pyramidal {b5['pyr'][0]:.1f}% ({b5['pyr'][1]}/{b5['pyr'][2]}), interneurons {b5['int'][0]:.1f}% "
              f"({b5['int'][1]}/{b5['int'][2]}); all four within tolerance under (b): {ok_b}")
allv = np.concatenate([[LR[k][i][3] <= 0.05 for i in range(S[k].n)] for k in S])
print(f"all cells, any type: {100 * allv.mean():.1f}% trajectory specific")

**Result: partially reproduced.** Over all sessions of each layer (reading b, B-a W-a), 70.2% of
layer 2/3 pyramidal cells, 55.6% of layer 2/3 interneurons and 85.7% of layer 5 interneurons are
trajectory specific, within tolerance, but 72.0% of layer 5 pyramidal cells, against 84.7 +- 8.47.
The layer 5 Figure 3 session is not in the data set.

### no-spatial-clustering

> Left- and right-specific neurons are not clustered on the probe but spread over the whole
> recorded volume.

Class per cell from its significant segments: left (left rate higher), right, both, or none.
Chi-square test of shank against class in the Figure 3 session (EE188), and each session as a
variant; Cramer's V.

In [ ]:
def lr_class(res):
    D0, point, glob, gp = res
    segs = pfc2.segments(point, glob)
    left = any(D0[a:b + 1].mean() > 0 for a, b in segs)
    right = any(D0[a:b + 1].mean() < 0 for a, b in segs)
    return "both" if left and right else "left" if left else "right" if right else "none"


def cluster_test(k):
    s = S[k]
    cls = np.array([lr_class(r) for r in LR[k]])
    labs = ["left", "right", "both", "none"]
    sh = np.unique(s.shank)
    tab = np.array([[np.sum((s.shank == h) & (cls == c)) for c in labs] for h in sh])
    tab = tab[:, tab.sum(0) > 0]
    tab = tab[tab.sum(1) > 0]
    chi2, p, dof, _ = stats.chi2_contingency(tab)
    v = np.sqrt(chi2 / (tab.sum() * (min(tab.shape) - 1)))
    return chi2, p, v, tab, cls


chi2, p, v, tab, cls = cluster_test("EE188")
print(f"EE188: classes {dict(zip(*np.unique(cls, return_counts=True)))}; shanks x classes table {tab.tolist()}; "
      f"chi-square {chi2:.1f}, p = {p:.3f}, Cramer's V = {v:.2f}; association at p < 0.05: {p < 0.05}")
for k in S:
    if k != "EE188":
        chi2, p, v, tab, cls = cluster_test(k)
        print(f"  {k}: chi-square {chi2:.1f}, p = {p:.3f}, V = {v:.2f}, {S[k].n} cells")

**Result: reproduced.** In EE188 the left, right, both and none classes do not depend on the shank
(chi-square p = 0.577, Cramer's V = 0.23). Among the other sessions (a variant), FF130 shows an
association (p = 0.047) and the other 11 do not.

## Position dependence of spike transmission

For each excitatory connection, coincident events (1 <= tdiff <= 4 ms) are placed at the
presynaptic spike's position within trials, smoothed with the 0.05 kernel and divided by the
number of trials, and compared with 1,000 surrogates jittering every spike of both cells on
[-5, +5] ms, one-sided for excess, global level 0.01 (Methods). Readings of the claims file:
trials (a) left and right separately, a pair counting when either shows dependence, (b) all
trials together; dependence (i) a globally significant region anywhere, (ii) a globally
significant region and a part of the maze where the pointwise band is not broken. The
connections are those of reading B-a W-a, which finds all 43 of the authors' `pairs_all`;
B-b W-a is a variant. Jittering the presynaptic cell only (Supplementary Figure 12B) was not
run.

In [ ]:
POSDEP = {}
t0 = time.time()
rng = np.random.default_rng(300)
pairs_needed = set()
for B in ("B-a", "B-b"):
    for k in S:
        E, _ = conns(k, B, "W-a")
        pairs_needed |= {(k, i, j) for i, j in zip(*np.nonzero(E))}
for k, i, j in sorted(pairs_needed):
    s = S[k]
    res = {}
    for name, m in (("L", s.left), ("R", ~s.left), ("all", np.ones(s.ntrials, bool))):
        S0, mu, point, glob, gp = pfc2.coincidence_test(s, i, j, rng, trials_mask=m)
        res[name] = (gp <= 0.01, bool(glob.any() and (~point).any()), point, glob, S0, mu)
    POSDEP[(k, i, j)] = res
print(f"coincidence tests on {len(POSDEP)} excitatory connections x 3 trial sets in {round(time.time() - t0)} s")


def posdep(key, trials, dep):
    r = POSDEP[key]
    sets = ("L", "R") if trials == "a" else ("all",)
    return any(r[t][0] if dep == "i" else r[t][1] for t in sets)

### position-dependent-transmission

> Of 343 pairs with significant excitatory monosynaptic connections, 67 show identifiable
> position dependence of excess short-latency coincidences.

In [ ]:
for B in ("B-a", "B-b"):
    keys = [(k, i, j) for k in S for i, j in zip(*np.nonzero(conns(k, B, "W-a")[0]))]
    for trials in "ab":
        for dep in ("i", "ii"):
            n = sum(posdep(x, trials, dep) for x in keys)
            ok = (abs(len(keys) - 343) <= 34.3 and abs(n - 67) <= 6.7) or abs(100 * n / len(keys) - 19.5) <= 2.0
            print(f"{B} W-a, trials {trials}, dependence {dep:2s}: {n} of {len(keys)} excitatory connections "
                  f"({100 * n / len(keys):.1f}%); counts or fraction within tolerance: {ok}")
keys = [(k, i, j) for k in S for i, j in zip(*np.nonzero(conns(k, "B-a", "W-a")[0]))]
for r in ("EE", "FF", "GG"):
    kk = [x for x in keys if S[x[0]].rat == r]
    print(f"  rat {r}: {sum(posdep(x, 'a', 'i') for x in kk)} of {len(kk)} (B-a, a, i)")
# null check: the same test with pre spikes shifted by 20 ms, which removes the monosynaptic lag
rng0 = np.random.default_rng(301)
sub = keys[::10]
fp = 0
for k, i, j in sub:
    s = S[k]
    ps = pfc2.pair_spikes(s, i, j)
    ps["tpre"] = ps["tpre"] + 0.02
    fp += pfc2.coincidence_test(s, i, j, rng0, ps=ps)[4] <= 0.01
print(f"null check, presynaptic spikes shifted by 20 ms: {fp} of {len(sub)} pairs significant (level 0.01)")

**Result: not reproduced.** Of 439 excitatory connections (B-a), 32 show position dependence (7.3%,
readings a and i), 32 to 35 under the other readings, against 67 of 343 (19.5%); B-b gives 31 to 36
of 387. Shifting the presynaptic spikes by 20 ms makes 0 of 44 pairs significant.

### fig5a-session-selective-connections

> In the layer 2/3 session of Figure 3d, 15 of 36 excitatory connections are active selectively in
> maze segments of left or right trajectories.

EE188, with the readings above. The authors' `pairs_all` (43 pairs) is the set drawn in Figure
3d (its largest component is 39 cells, the paper's 39); its excitatory connections are a variant
set.

In [ ]:
for B in ("B-a", "B-b"):
    kk = [("EE188", i, j) for i, j in zip(*np.nonzero(conns("EE188", B, "W-a")[0]))]
    for trials in "ab":
        for dep in ("i", "ii"):
            n = sum(posdep(x, trials, dep) for x in kk)
            print(f"{B} W-a, trials {trials}, dependence {dep:2s}: {n} of {len(kk)} connections; within 15 +- 1.5 and "
                  f"36 +- 3.6: {abs(n - 15) <= 1.5 and abs(len(kk) - 36) <= 3.6}")
kk = [("EE188", i, j) for i, j in zip(*np.nonzero(PA))]
for trials in "ab":
    for dep in ("i", "ii"):
        n = sum(posdep(x, trials, dep) for x in kk if x in POSDEP)
        print(f"pairs_all ({len(kk)} pairs), trials {trials}, dependence {dep}: {n}")

**Result: not reproduced.** In EE188, 6 or 7 of 105 excitatory connections (B-a) and 7 to 10 of 88
(B-b) show position dependence, against 15 of 36; of the authors' 43 `pairs_all` connections, 5 or
6.

### fig4-pair-156-136

> Pair 156-136, on different shanks, keeps steady firing rates between positions 0.2 and 0.5 on
> left trials, yet excess short-latency coincidences occur only between 0.4 and 0.6.

Cell numbers are the data set's (`CellsInfo` of `EE188_example.mat`, the numbering of the
paper's figures by the data description).

In [ ]:
def example_pair(a, b, trials, seed):
    s = S["EE188"]
    i, j = idx[a], idx[b]
    m = s.left if trials == "L" else ~s.left
    S0, mu, point, glob, gp = pfc2.coincidence_test(s, i, j, np.random.default_rng(seed), trials_mask=m)
    segs = [(round(G[x] - 0.5 / pfc2.NGRID, 3), round(G[y] + 0.5 / pfc2.NGRID, 3)) for x, y in pfc2.segments(point, glob)]
    gl = np.flatnonzero(glob)
    gext = (round(G[gl[0]], 3), round(G[gl[-1]], 3)) if len(gl) else None
    return segs, gext, gp, S0, mu


s = S["EE188"]
print("shanks: 156 on", s.shank[idx[156]], ", 136 on", s.shank[idx[136]], "; types (CellsInfo)",
      s.celltype_file[idx[156]], s.celltype_file[idx[136]])
E, I = conns("EE188", "B-a", "W-a")
print("connection 156 -> 136 (B-a W-a): excitatory", E[idx[156], idx[136]], "; in pairs_all:", PA[idx[156], idx[136]])
segs, gext, gp, S0, mu = example_pair(156, 136, "L", 401)
print(f"left trials: global p = {gp:.4f}; significant regions (pointwise extent) {segs}; global band broken "
      f"from {gext[0] if gext else None} to {gext[1] if gext else None}")
PLp = H["EE188"][idx[156]][s.left].mean(0)
PLq = H["EE188"][idx[136]][s.left].mean(0)
r25 = (G >= 0.2) & (G <= 0.5)
for c, P in ((156, PLp), (136, PLq)):
    v = P[r25]
    print(f"cell {c} left PSTH 0.2-0.5: {v.min():.1f} to {v.max():.1f} (spikes per unit length per trial); "
          f"variation {(v.max() - v.min()) / v.max() * 100:.0f}% of the maximum (steady if < 50%)")
if segs:
    st, en = segs[0]
    print(f"only one region: {len(segs) == 1}; start {st} within 0.4 +- 0.04: {abs(st - 0.4) <= 0.04}; end {en} within "
          f"0.6 +- 0.06: {abs(en - 0.6) <= 0.06}")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(G, S0, label="coincidences")
ax.plot(G, mu, label="jitter mean")
ax.set_xlabel("normalised position")
ax.set_ylabel("events per unit length per trial")
ax.legend()
ax.set_title("156-136, left trials")
plt.show()
print(f"plotted range: coincidences {S0.min():.2f} to {S0.max():.2f}, jitter mean {mu.min():.2f} to {mu.max():.2f}")

**Result: not reproduced.** On left trials the excess coincidences of 156-136 form one significant
region, from 0.325 to 0.645 by the pointwise band (global band broken from 0.372 to 0.622); the end
is within 0.6 +- 0.06 but the start is not within 0.4 +- 0.04, and cell 136's rate between 0.2 and
0.5 varies by 59% of its maximum (cell 156 by 49%).

### fig5b-example-pairs

> In pair 197-201 transmission is significant only between 0.4 and 0.5 of left trials although
> the expected coincidence rate is as high from 0 to 0.5; pair 49-52 transmits only on left
> trials; pair 186-201 only toward the end of the right arm.

In [ ]:
res5b = {}
for (a, b), seed in (((197, 201), 402), ((49, 52), 403), ((186, 201), 404)):
    for tr in ("L", "R"):
        segs, gext, gp, S0, mu = example_pair(a, b, tr, seed)
        res5b[(a, b, tr)] = segs
        e0 = mu[(G >= 0) & (G <= 0.5)]
        print(f"{a}-{b} {tr}: global p = {gp:.4f}, regions {segs}, global band broken {gext}; jitter mean (expected "
              f"coincidences) 0-0.5 from {e0.min():.2f} to {e0.max():.2f}; excitatory connection (B-a W-a) "
              f"{E[idx[a], idx[b]]}")
s1 = res5b[(197, 201, "L")]
c1 = len(s1) == 1 and abs(s1[0][0] - 0.4) <= 0.04 and abs(s1[0][1] - 0.5) <= 0.05
c2 = len(res5b[(49, 52, "L")]) > 0 and len(res5b[(49, 52, "R")]) == 0
c3 = len(res5b[(186, 201, "R")]) == 1 and res5b[(186, 201, "R")][0][0] > 0.75 and len(res5b[(186, 201, "L")]) == 0
c3b = len(res5b[(186, 201, "R")]) >= 1 and all(x[0] > 0.75 for x in res5b[(186, 201, "R")])
print(f"197-201 meets its expected values: {c1}; 49-52 left only: {c2}; 186-201 only region on right trials beyond 0.75: "
      f"{c3} (right-trial regions all beyond 0.75: {c3b}); pairs holding: {c1 + c2 + c3}")

**Result: not reproduced.** 197-201 has one significant region on left trials, from 0.355 to 0.525
(expected 0.4 +- 0.04 to 0.5 +- 0.05; the start misses by 0.005), and none on right trials; 49-52
has none on either side (global p 0.55 and 0.80); 186-201 has none on left trials and one on right
trials from 0.715 to 0.975 (global band from 0.752), which begins before 0.75. None of the three
pairs meets the criterion as written.

### thinning-preserves-dependence

> In several pairs, position-dependent transmission remains after spikes are thinned so that both
> cells fire at a flat rate over the region examined.

Methods, "Controlling for power": in the region, spikes are removed one at a time at random within
[-s, +s] of the PSTH maximum until the PSTH minimum is within 10% of the maximum, for both cells,
then the position test. Pairs: the position-dependent connections (B-a W-a, trials a, dependence
i), on the trials where they are dependent. Regions: (a) 0.2 to 0.5 (Supplementary Figure 13A),
(b) the smallest span holding the pair's significant region plus an equal length of
non-significant maze next to it. Ten repetitions with fixed seeds; a pair holds when, in the
majority of repetitions, a globally significant region and a non-significant part of the region
remain.

In [ ]:
def thin_pair(key, tr, lo, hi, rng):
    k, i, j = key
    s = S[k]
    m = s.left if tr == "L" else ~s.left
    ps = pfc2.pair_spikes(s, i, j, m)
    kp = pfc2.thin(ps["tpre"], ps["xpre"], ps["inpre"], lo, hi, rng)
    kq = pfc2.thin(ps["tpost"], ps["xpost"], ps["inpost"], lo, hi, rng)
    a = kp | ~ps["inpre"]
    b = kq | ~ps["inpost"]
    p2 = {"tpre": ps["tpre"][a], "xpre": ps["xpre"][a], "inpre": kp[a], "tpost": ps["tpost"][b], "N": ps["N"]}
    S0, mu, point, glob, gp = pfc2.coincidence_test(s, i, j, rng, ps=p2)
    reg = (G >= lo) & (G <= hi)
    return bool((glob & reg).any() and (~point & reg).any())


rng = np.random.default_rng(500)
cand = []
for key in keys:
    for tr in ("L", "R"):
        if POSDEP[key][tr][0]:
            cand.append((key, tr))
print(f"{len(cand)} position-dependent connection and trial-set combinations (B-a W-a)")
THIN = {}
for key, tr in cand:
    point, glob = POSDEP[key][tr][2], POSDEP[key][tr][3]
    gl = np.flatnonzero(glob)
    a0, a1 = G[gl[0]], G[gl[-1]]
    span = a1 - a0 + 0.05
    lo_b, hi_b = (a0 - span, a1 + 0.025) if a0 - span >= 0 else (a0 - 0.025, min(a1 + span, 1.0))
    for reading, (lo, hi) in (("a", (0.2, 0.5)), ("b", (max(lo_b, 0.0), hi_b))):
        if reading == "a" and not (glob & (G >= 0.2) & (G <= 0.5)).any():
            continue
        THIN[(key, tr, reading)] = [thin_pair(key, tr, lo, hi, rng) for rep in range(10)]
for reading in "ab":
    v = {x: sum(r) for x, r in THIN.items() if x[2] == reading}
    held = sum(n >= 6 for n in v.values())
    print(f"reading {reading}: {len(v)} pairs tested, {held} keep a significant and a non-significant part in the "
          f"majority of 10 thinnings; counts per pair {sorted(v.values(), reverse=True)}; at least two: {held >= 2}")

**Result: reproduced.** In the region 0.2 to 0.5 (reading a), 2 of 16 position-dependent connections
keep a significant region and a non-significant part in the majority of 10 thinnings (10 and 6 of
10); with the region around each significant region (reading b), 1 of 37.

## Spike order and convergence

### fig6-depression-facilitation

> Comparing first spikes of trains with later spikes, about equal shares of pairs show depressing
> (12.7%) and facilitating (10.7%) spike transmission; 32.2% of the significant pairs are on
> different shanks.

Presynaptic spikes are first (at least 200 ms without a spike before) or later (within 40 ms of
the previous spike). For each group, the standardised peak height against jitter surrogates
(100, jittering both cells on [-5, +5] ms); the difference, first minus later, is tested by
shuffling the labels between the groups (500 permutations), two-sided at P < 0.10. The numbers
of surrogates and permutations are standard choices made smaller than 1,000 for time. Readings:
pairs (a) the excitatory connections, (b) all connections; height (i) largest 1 ms bin in
[1, 4] ms, (ii) sum over [1, 4] ms; later spikes (x) in a train begun by a first spike, (y) any
spike within 40 ms. Connections of B-a W-a.

In [ ]:
t0 = time.time()
rng = np.random.default_rng(600)
F6 = {}
for k, s in S.items():
    E, I = conns(k, "B-a", "W-a")
    for i, j in zip(*np.nonzero(E | I)):
        tpre, tpost = s.spikes_of(i), s.spikes_of(j)
        for ch in ("x", "y"):
            f, l = pfc2.train_labels(tpre, chained=(ch == "x"))
            if f.sum() < 5 or l.sum() < 5:
                continue
            for how in ("i", "ii"):
                F6[(k, i, j, ch, how)] = pfc2.order_effect(tpre, tpost, f, l, rng, how, nsur=100, nperm=500) + (bool(E[i, j]),)
print(f"order effects for {len(F6)} pair-reading combinations in {round(time.time() - t0)} s")
for pairs in "ab":
    for ch in ("x", "y"):
        for how in ("i", "ii"):
            rows = [(key, v) for key, v in F6.items() if key[3] == ch and key[4] == how and (pairs == "b" or v[6])]
            ntot = sum(1 for k in S for _ in zip(*np.nonzero(conns(k, "B-a", "W-a")[0] if pairs == "a" else
                                                            (conns(k, "B-a", "W-a")[0] | conns(k, "B-a", "W-a")[1]))))
            dep = [key for key, v in rows if v[3] < 0.10 and v[2] > 0]
            fac = [key for key, v in rows if v[3] < 0.10 and v[2] < 0]
            sig = dep + fac
            diff = [key for key in sig if S[key[0]].shank[key[1]] != S[key[0]].shank[key[2]]]
            pd, pf_ = 100 * len(dep) / ntot, 100 * len(fac) / ntot
            psh = 100 * len(diff) / max(len(sig), 1)
            ok = abs(pd - 12.7) <= 1.27 and abs(pf_ - 10.7) <= 1.07 and abs(psh - 32.2) <= 3.22
            print(f"pairs {pairs}, later {ch}, height {how:2s}: {len(rows)} of {ntot} pairs testable; depression {len(dep)} "
                  f"({pd:.1f}%), facilitation {len(fac)} ({pf_:.1f}%); significant pairs on different shanks {len(diff)} "
                  f"({psh:.1f}%); all three within tolerance: {ok}")
# null rate: labels assigned at random to the same numbers of spikes
rng0 = np.random.default_rng(601)
nul = []
for key in [x for x in F6 if x[3] == "x" and x[4] == "i"][::5]:
    k, i, j = key[:3]
    tpre, tpost = S[k].spikes_of(i), S[k].spikes_of(j)
    f, l = pfc2.train_labels(tpre)
    sel = np.flatnonzero(f | l)
    perm = rng0.permutation(sel)
    f2 = np.zeros(len(tpre), bool)
    l2 = np.zeros(len(tpre), bool)
    f2[perm[:f.sum()]] = True
    l2[perm[f.sum():]] = True
    nul.append(pfc2.order_effect(tpre, tpost, f2, l2, rng0, "i", nsur=100, nperm=500)[3] < 0.10)
print(f"null check, labels assigned at random: {np.sum(nul)} of {len(nul)} pairs significant at P < 0.10")

**Result: not reproduced.** Depressing pairs are 4.8 to 9.1% and facilitating pairs 3.0 to 8.2% of
the connections, against 12.7% and 10.7%; 29.0 to 34.0% of the significant pairs are on different
shanks, within 32.2 +- 3.22 under every reading. With labels assigned at random, 18 of 121 pairs
(15%) are significant at P < 0.10.

### fig6b-convergent-examples

> Two putative pyramidal cells converging on one layer 2/3 interneuron behave oppositely; cell 197
> depresses with spike repetition while the first spikes of cell 186 are ineffective and its
> later spikes drive the interneuron.

In [ ]:
s = S["EE188"]
ok197 = ok186 = False
for ch in ("x", "y"):
    for how in ("i", "ii"):
        r1 = F6.get(("EE188", idx[197], idx[201], ch, how))
        r2 = F6.get(("EE188", idx[186], idx[201], ch, how))
        for name, r in (("197-201", r1), ("186-201", r2)):
            if r is None:
                print(f"{name} later {ch} height {how}: not a connection under B-a W-a")
                continue
            print(f"{name} later {ch} height {how}: first z = {r[0]:.2f} ({r[4]} spikes), later z = {r[1]:.2f} ({r[5]} spikes), "
                  f"difference {r[2]:.2f}, P = {r[3]:.3f}")
        ok197 |= r1 is not None and r1[2] > 0 and r1[3] < 0.10
        ok186 |= r2 is not None and r2[2] < 0 and r2[3] < 0.10
print("CellsInfo types: 197", s.celltype_file[idx[197]], "186", s.celltype_file[idx[186]], "201", s.celltype_file[idx[201]])
print(f"197-201 depressing at P < 0.10 under one reading: {ok197}; 186-201 facilitating at P < 0.10 under one reading: {ok186}")

**Result: partially reproduced.** 186-201 facilitates: the later spikes drive 201 more than the
first (difference -8.21, no permutation of 500 as large, with later spikes as any spike within 40
ms; -4.35, P = 0.162 in chained trains). 197-201 has a higher first-spike height than later
(difference 4.71 to 8.13), as in the paper, but not significantly (P = 0.454 to 0.648).

### fig7-supralinear-coincidence

> Coincident spikes of two or more presynaptic cells within 5 ms discharge their common target
> interneuron supralinearly, more than the sum of their separate effects, while coincidences in
> windows over 10 ms add only linearly.

A satellite is a putative interneuron (B-a W-a, C-a) receiving excitatory connections from two or
more putative pyramidal cells. An event is a presynaptic spike with k distinct presynaptic cells
spiking in the window w ending at it, and no presynaptic spike in the 4 ms after it; it transmits
when the target fires 1 to 4 ms after it. p1 is from events with k = 1 (no other presynaptic cell
within w before or 4 ms after). Supralinear: (a) p_k > k p1, (b) p_k > 1 - (1 - p1)^k. Events
with too few occurrences (under 20) give no value.

In [ ]:
def satellite_curve(s, pres, tgt, w):
    tl = [s.spikes_of(p) for p in pres]
    t = np.concatenate(tl)
    c = np.concatenate([np.full(len(x), n) for n, x in enumerate(tl)])
    o = np.argsort(t)
    t, c = t[o], c[o]
    tt = s.spikes_of(tgt)
    nxt = np.append(np.diff(t), np.inf)
    a = np.searchsorted(t, t - w, side="right")
    k = np.array([len(np.unique(c[a[m]:m + 1])) for m in range(len(t))])
    i0 = np.searchsorted(tt, t + 0.001, side="left")
    i1 = np.searchsorted(tt, t + 0.004, side="right")
    hit = i1 > i0
    clean = nxt > 0.004
    prev_other = np.array([np.any(c[a[m]:m] != c[m]) for m in range(len(t))])
    p = {}
    for kk in (1, 2, 3):
        m = clean & (k == kk) & ((~prev_other) if kk == 1 else True)
        p[kk] = (hit[m].mean() if m.sum() >= 20 else np.nan, int(m.sum()))
    return p


SAT = []
for k, s in S.items():
    E, I = conns(k, "B-a", "W-a")
    pyr, inn = TYPES[(k, "B-a", "W-a", "C-a")]
    for tgt in np.flatnonzero(inn):
        pres = np.flatnonzero(E[:, tgt] & pyr)
        if len(pres) >= 2:
            SAT.append((k, tgt, pres))
print(f"satellites: {len(SAT)} (paper 14); within 14 +- 1.4: {abs(len(SAT) - 14) <= 1.4}; presynaptic cells per satellite "
      f"{[len(x[2]) for x in SAT]}")
for w in (0.002, 0.005, 0.01, 0.02):
    rows = []
    for k, tgt, pres in SAT:
        p = satellite_curve(S[k], pres, tgt, w)
        rows.append([p[1][0], p[2][0], p[3][0], p[2][1], p[3][1]])
    R = np.array(rows, float)
    p1, p2, p3 = R[:, 0], R[:, 1], R[:, 2]
    for kk, pk in ((2, p2), (3, p3)):
        ok = ~np.isnan(pk) & ~np.isnan(p1)
        lin_a = kk * p1[ok]
        lin_b = 1 - (1 - p1[ok]) ** kk
        ex_a = pk[ok] - lin_a
        ex_b = pk[ok] - lin_b
        print(f"window {w * 1000:4.0f} ms, k = {kk}: {ok.sum()} satellites with values; mean p{kk} {np.mean(pk[ok]):.3f}, "
              f"mean p1 {np.mean(p1[ok]):.3f}; above k*p1 in {np.sum(ex_a > 0)}, mean excess {np.mean(ex_a):.3f} "
              f"(s.d. {np.std(ex_a, ddof=1) if ok.sum() > 1 else np.nan:.3f}); above 1-(1-p1)^k in {np.sum(ex_b > 0)}, "
              f"mean excess {np.mean(ex_b):.3f} (s.d. {np.std(ex_b, ddof=1) if ok.sum() > 1 else np.nan:.3f})")

**Result: not reproduced.** There are 15 satellites (14 in the paper). With 5 ms windows the
transmission probability after two coincident presynaptic cells (mean 0.058) is about that after one
(0.053), below k p1 and 1 - (1 - p1)^k in every satellite, and the same holds for three cells and
for every window.

## Summary

| Claim | Outcome |
|---|---|
| unit-count | reproduced |
| task-performance | blocked (data missing: correct trials only) |
| fig2c-synchrony-by-layer | partially reproduced |
| fig2-population-uniform | not reproduced |
| lifetime-width | reproduced |
| trajectory-divergence | not reproduced |
| side-arm-selectivity-by-layer | not reproduced |
| central-arm-selectivity | not reproduced |
| odor-selective-fraction | not reproduced |
| odor-unrelated-to-stem | reproduced |
| retrospective-coding-rare | reproduced |
| connection-counts | not reproduced |
| cell-type-fractions | not reproduced |
| pyr-pyr-connections | not reproduced |
| fig3b-session-connections | not reproduced |
| fig3d-hub | not reproduced |
| fig3f-distance | partially reproduced |
| nearby-pyr-int-probability | not reproduced |
| connected-cells-trajectory-specific | partially reproduced |
| no-spatial-clustering | reproduced |
| position-dependent-transmission | not reproduced |
| fig5a-session-selective-connections | not reproduced |
| fig4-pair-156-136 | not reproduced |
| fig5b-example-pairs | not reproduced |
| thinning-preserves-dependence | reproduced |
| fig6-depression-facilitation | not reproduced |
| fig6b-convergent-examples | partially reproduced |
| fig7-supralinear-coincidence | not reproduced |
| speed-not-correlated | reproduced |
| population-vector-width | reproduced (resting on an assumption: PSTHs normalised per cell) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The data set holds three of the paper's four rats
(EE, FF, GG, taken to be the paper's E, F and G; rat H is absent), only correct trials, and, for
each session, a selection of the cells (`CellsInfo`; the files list 1,495 clusters, 627 cells).
The layer 5 session of Figure 2a and Supplementary Figure 11 (142 cells) is not in the data set,
so every layer 5 value of a single-session claim is untested. The number of permutations and
surrogates is 1,000 except for the spike-order test (100 surrogates, 500 permutations). The
global band of the jitter test takes 1% on each side, a reading of the Methods. The detected
connections include all 43 of the authors' `pairs_all` for EE188 (G-a B-a), but the fraction of
connected pairs is 1.4 to 1.9 times the paper's (1.05 to 1.5 times with the level split between
the sides, G-b), and the counts that follow from them (cell types, hub,
connection probabilities) differ accordingly.

What was found about the data set. The data description names `EE188_example.mat` as a file of
folder EE0708; it is a member of `EE.188.tar.gz`. The `CellsInfo` type column, described as
pyramidal, interneuron or unknown, holds no unknown cell in the 12 sessions of `behaviors.zip` or
in `EE188_Behavior_PFC.mat` (87 and 18 in EE188), while the example file of the same session
gives 33, 15 and 69; how either was derived is not documented. `EE188_Behavior_PFC.mat` holds 105
of the example's 117 cells. The spikes' trial labels and positions run from the first nose-poke,
while the description defines the trial start as the end of the nose-poke. In 10 of 13 sessions
the second LED is not tracked, so head direction is available only in EE152, EE165 and EE188.
The layer of each rat is not given in the data set; it comes from the paper's Supplementary
Figure 1.

In [ ]:
print(f"notebook run time {(time.time() - T0) / 60:.1f} min")